# 深度循环神经网络（Deep RNN）

## 什么是深度RNN？

就像卷积神经网络可以有多个卷积层一样，循环神经网络也可以堆叠多层，形成**深度RNN**。

### 单层RNN vs 多层RNN

**单层RNN**：
```
输入 → RNN层 → 输出
```

**深度RNN**（以2层为例）：
```
输入 → RNN层1 → RNN层2 → 输出
       ↓         ↓
     隐状态1   隐状态2
```

### 深度RNN的优势

1. **更强的表示能力**：多层结构可以学习更复杂的模式
2. **层次化特征**：底层捕捉简单特征，高层捕捉抽象特征
3. **更好的性能**：在许多任务上，2-4层的深度RNN表现最好

### 注意事项

- **过深的问题**：RNN层数过多（>4层）可能导致训练困难
- **参数量增加**：每增加一层，参数量显著增加
- **计算成本**：训练时间会明显增长

## 本代码实现

本代码构建一个2层的深度LSTM，用于字符级语言建模任务。

## 步骤1：导入库和加载数据

In [ ]:
# ==================== 导入必要的库 ====================
import torch
import RNN  # 自定义模块，包含数据加载、模型定义和训练函数
from torch import nn
from d2l import torch as d2l

# ==================== 设置数据参数 ====================
# batch_size = 32: 每次训练使用32个序列样本
# num_steps = 35: 每个序列包含35个时间步（35个字符）
# 这些参数会影响：
#   - 训练速度：batch_size越大，训练越快，但需要更多内存
#   - 序列长度：num_steps越大，能捕获更长的依赖关系
batch_size, num_steps = 32, 35

# ==================== 加载时间机器数据集 ====================
# "时间机器"是H.G.威尔斯的经典科幻小说
# train_iter: 数据迭代器，每次返回一批序列数据
# vocab: 词汇表对象，存储字符与索引的映射关系
train_iter, vocab = RNN.load_data_time_machine(batch_size, num_steps)

## 步骤2：构建深度LSTM模型

这一步创建一个2层的深度LSTM网络。关键参数 `num_layers=2` 使得模型具有两层LSTM结构。

In [ ]:
# ==================== 定义模型超参数 ====================
# vocab_size: 词汇表大小，即不同字符的数量（通常是28个：26个字母+空格+特殊字符）
# num_hiddens: 隐藏层维度为256，控制每层LSTM的神经元数量
#              - 越大：模型容量越强，但训练越慢，容易过拟合
#              - 越小：训练快，但表达能力弱
# num_layers: 层数为2，这是关键！表示堆叠2层LSTM
#             数据流向：输入 → LSTM层1 → LSTM层2 → 输出
vocab_size, num_hiddens, num_layers = len(vocab), 256, 2
num_inputs = vocab_size

# ==================== 选择计算设备 ====================
# 尝试使用GPU加速，如果没有GPU则使用CPU
device = d2l.try_gpu()

# ==================== 创建深度LSTM层 ====================
# nn.LSTM参数说明：
#   - num_inputs: 输入特征维度（词汇表大小）
#   - num_hiddens: 每层的隐藏单元数
#   - num_layers: LSTM层数（2层）
# 注意：这里没有设置 bidirectional=True，所以是单向的深度LSTM
lstm_layer = nn.LSTM(num_inputs, num_hiddens, num_layers)

# ==================== 封装为完整模型 ====================
# RNNModel是一个包装类，将LSTM层和输出层组合起来
# vocab_size: 输出层大小，用于预测下一个字符
model = RNN.RNNModel(lstm_layer, vocab_size).to(device)

# 再次确保模型在正确的设备上（这行其实有些冗余）
model = model.to(device)

## 步骤3：训练模型

使用准备好的数据和模型进行训练。训练过程中会显示困惑度（perplexity）的变化，困惑度越低说明模型性能越好。

In [ ]:
# ==================== 设置训练超参数 ====================
# num_epochs = 500: 训练500个epoch（完整遍历数据集500次）
#                    - 对于小数据集，需要多次迭代才能充分学习
# lr = 2: 学习率为2
#         - 这个学习率相对较大
#         - 深度RNN使用更大的学习率（相比单层RNN的lr=1）
#         - 因为梯度需要经过多层传播，较大的学习率能加快收敛
num_epochs, lr = 500, 2

# ==================== 开始训练 ====================
# train_ch8函数会：
#   1. 在每个epoch中遍历所有训练数据
#   2. 计算损失并反向传播更新参数
#   3. 定期输出训练进度和困惑度（perplexity）
#   4. 生成示例文本，检查模型的生成质量
# 
# 困惑度（Perplexity）是语言模型的常用评估指标：
#   - 困惑度 = exp(交叉熵损失)
#   - 越低越好，表示模型预测越准确
#   - 例如困惑度为10，意味着模型平均在10个候选字符中犹豫
RNN.train_ch8(model, train_iter, vocab, lr, num_epochs, device)